# SHL Grammar Scoring Engine

A reproducible development notebook for SHL Hiring Assessment 2026.

**Stage 1:** full-recording acoustic features with supervised regression.
**Stage 2:** transcribed speech and content-aware features, retained only if validation improves.

Only the supplied training labels are used. Test labels are placeholders.
The provided sample submission has stale filenames; outputs use `test.csv`.
Private recordings, labels and transcripts must not be published in a public repository.

This notebook displays the compulsory fitted-training RMSE separately from honest held-out-fold metrics. No leaderboard threshold or score is assumed.


## Setup

Run in a private Kaggle notebook with the competition attached. The acoustic baseline requires no GPU or external model downloads.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.signal import resample_poly
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from threadpoolctl import threadpool_limits
import joblib
import matplotlib.pyplot as plt

SEED = 42
OUT = Path('/kaggle/working/shl_results')
OUT.mkdir(parents=True, exist_ok=True)
SR = 16000

## Data audit

Check schemas, identifiers, all audio paths and the actual 0–5 target range before modeling. Never use test placeholder labels as targets.


In [ ]:
roots = [p.parent for p in Path('/kaggle/input').rglob('train.csv')
         if (p.parent / 'test.csv').is_file()]
if len(roots) != 1:
    raise ValueError(f'Attach exactly one SHL dataset. Found {len(roots)} candidates.')
DATA_ROOT = roots[0]
train = pd.read_csv(DATA_ROOT / 'train.csv')
test = pd.read_csv(DATA_ROOT / 'test.csv')
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
for split, frame in [('train', train), ('test', test)]:
    if list(frame.columns) != ['filename', 'label']:
        raise ValueError(f'Unexpected {split} columns: {list(frame.columns)}')
    if frame.filename.isna().any() or frame.filename.duplicated().any():
        raise ValueError(f'Missing or duplicate identifiers in {split}.')
    if any(Path(name).name != name for name in frame.filename):
        raise ValueError('Audio identifiers must be filenames, not paths.')
    missing = [name for name in frame.filename
               if not (DATA_ROOT / split / name).is_file()]
    if missing:
        raise FileNotFoundError(f'{len(missing)} missing {split} recordings.')
y = pd.to_numeric(train.label, errors='raise').to_numpy(dtype=np.float64)
assert np.isfinite(y).all() and ((y >= 0) & (y <= 5)).all()
audit = {
    'training_rows': len(train), 'test_rows': len(test), 'sample_rows': len(sample),
    'sample_test_id_overlap': len(set(sample.filename) & set(test.filename)),
    'sample_ids_absent_from_test': len(set(sample.filename) - set(test.filename)),
    'training_label_counts': {str(k): int(v) for k, v in
                              train.label.value_counts().sort_index().items()},
    'submission_identifiers': 'test.csv, in original order',
}
print(json.dumps(audit, indent=2), flush=True)
if set(sample.filename) != set(test.filename):
    print('The supplied sample has incompatible identifiers; retain its columns '
          'and use the current test.csv identifiers.', flush=True)
(OUT / 'data_audit.json').write_text(json.dumps(audit, indent=2))

## Audio features

Use complete recordings. Resample to mono 16 kHz, summarize MFCCs, spectral measures and pauses over time. These are a quick reference baseline; grammar is fundamentally linguistic.


In [ ]:
def read_audio(path):
    """Read the entire recording as mono float32 and resample to 16 kHz."""
    audio, original_sr = sf.read(str(path), dtype='float32', always_2d=True)
    if len(audio) == 0 or not np.isfinite(audio).all():
        raise ValueError(f'Empty/nonfinite audio: {path.name}')
    audio = audio.mean(axis=1)
    if original_sr != SR:
        divisor = math.gcd(original_sr, SR)
        audio = resample_poly(audio, SR // divisor, original_sr // divisor)
    return np.ascontiguousarray(audio, dtype=np.float32)


def summarize(matrix):
    """Describe variation through time, rather than taking one short segment."""
    return np.concatenate([
        matrix.mean(axis=1), matrix.std(axis=1),
        np.quantile(matrix, .10, axis=1), np.median(matrix, axis=1),
        np.quantile(matrix, .90, axis=1),
    ])


def acoustic_features(path):
    audio = read_audio(path)
    duration = len(audio) / SR
    audio = np.pad(audio, (0, max(0, 1024 - len(audio))))
    spectrum = np.abs(librosa.stft(audio, n_fft=512, hop_length=256))
    power = spectrum ** 2
    mel = librosa.feature.melspectrogram(S=power, sr=SR, n_fft=512,
                                        n_mels=40, fmin=40, fmax=7800)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel, ref=1.0), n_mfcc=20)
    delta = librosa.feature.delta(mfcc, mode='nearest')
    rms = librosa.feature.rms(S=spectrum, frame_length=512)
    zcr = librosa.feature.zero_crossing_rate(audio, frame_length=512,
                                            hop_length=256)
    spectral = np.vstack([
        librosa.feature.spectral_centroid(S=spectrum, sr=SR),
        librosa.feature.spectral_bandwidth(S=spectrum, sr=SR),
        librosa.feature.spectral_rolloff(S=spectrum, sr=SR, roll_percent=.85),
        librosa.feature.spectral_flatness(S=spectrum), rms, zcr,
    ])
    speech = rms[0] > max(1e-5, .10 * float(np.quantile(rms, .95)))
    pauses = ~speech
    starts = int(np.sum(pauses[1:] & ~pauses[:-1]))
    extra = [duration, float(np.max(np.abs(audio))),
             float(np.sqrt(np.mean(audio ** 2))), float(speech.mean()),
             starts / max(duration, 1e-6)]
    features = np.concatenate([summarize(mfcc), summarize(delta),
                               summarize(spectral), extra])
    if not np.isfinite(features).all():
        raise ValueError(f'Nonfinite extracted features: {path.name}')
    return features.astype(np.float32)


manifest = [(split, name) for split, frame in [('train', train), ('test', test)]
            for name in frame.filename]
paths = [DATA_ROOT / split / name for split, name in manifest]
cache = OUT / 'acoustic_features.npz'
manifest_id = hashlib.sha256(json.dumps(manifest).encode()).hexdigest()
started = time.time()
with threadpool_limits(limits=1):
    if cache.exists():
        cached = np.load(cache, allow_pickle=False)
        if str(cached['manifest_id']) != manifest_id:
            raise ValueError('Feature cache belongs to a different file manifest.')
        all_features = cached['features']
    else:
        print('Extracting full-recording audio features...', flush=True)
        # Warm up librosa before concurrent calls; restrict BLAS oversubscription.
        first = acoustic_features(paths[0])
        rows = [first]
        with ThreadPoolExecutor(max_workers=2) as pool:
            for i, features in enumerate(pool.map(acoustic_features, paths[1:]), 2):
                rows.append(features)
                if i % 50 == 0:
                    print(f'Audio features {i}/{len(paths)}; '
                          f'{time.time() - started:.0f}s elapsed', flush=True)
        all_features = np.vstack(rows)
        np.savez_compressed(cache, features=all_features, manifest_id=manifest_id)
print('Feature matrix:', all_features.shape, flush=True)
X, X_test = all_features[:len(train)], all_features[len(train):]

## Validation and model comparison

Five held-out folds compare models against a constant prediction. Exact duplicate recordings are kept together when present. Scaling is fitted inside each fold. These development folds are also used for model selection, so they are not an independent final test.


In [ ]:
def metrics(target, prediction):
    target, prediction = np.asarray(target), np.asarray(prediction)
    correlation = (float(np.corrcoef(target, prediction)[0, 1])
                   if target.std() > 1e-12 and prediction.std() > 1e-12 else None)
    return {'RMSE': float(np.sqrt(mean_squared_error(target, prediction))),
            'Pearson': correlation}


# Exact duplicate training files must never appear on opposite sides of a fold.
groups = np.array([hashlib.sha256(path.read_bytes()).hexdigest()
                   for path in paths[:len(train)]])
duplicate_count = len(groups) - len(set(groups))
if duplicate_count:
    splitter = GroupKFold(n_splits=5)
    folds = list(splitter.split(X, y, groups))
    validation_type = '5-fold validation grouped by exact recording hash'
else:
    bins = np.digitize(y, [.5, 2.5, 3.5, 4.5])
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    folds = list(splitter.split(X, bins))
    validation_type = '5-fold validation stratified by broad score intervals'
print('Validation:', validation_type, '; duplicate training files:', duplicate_count,
      flush=True)

models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=100.0)),
    'SVR': make_pipeline(StandardScaler(), SVR(C=10, epsilon=.15)),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=300, min_samples_leaf=3,
                                      max_features=.8, random_state=SEED, n_jobs=2),
    'HistogramBoosting': HistGradientBoostingRegressor(
        max_iter=250, max_leaf_nodes=15, l2_regularization=5,
        learning_rate=.05, random_state=SEED),
}
comparison, oof_predictions = [], {}
with threadpool_limits(limits=1):
    for name, estimator in models.items():
        oof = np.full(len(y), np.nan)
        fold_rmse = []
        for fit_ids, valid_ids in folds:
            fitted = clone(estimator).fit(X[fit_ids], y[fit_ids])
            prediction = np.clip(fitted.predict(X[valid_ids]), 0, 5)
            oof[valid_ids] = prediction
            fold_rmse.append(metrics(y[valid_ids], prediction)['RMSE'])
        assert np.isfinite(oof).all()
        row = {'model': name, **metrics(y, oof),
               'fold_RMSE_sd': float(np.std(fold_rmse))}
        comparison.append(row)
        oof_predictions[name] = oof
        print(row, flush=True)
scores = pd.DataFrame(comparison).sort_values('RMSE').reset_index(drop=True)
scores.to_csv(OUT / 'validation_metrics.csv', index=False)
best_name = scores.iloc[0]['model']
best_oof = oof_predictions[best_name]
with threadpool_limits(limits=1):
    best_model = clone(models[best_name]).fit(X, y)
    train_prediction = np.clip(best_model.predict(X), 0, 5)
    test_prediction = np.clip(best_model.predict(X_test), 0, 5)
training_metrics = metrics(y, train_prediction)
validation_metrics = metrics(y, best_oof)
print('Selected acoustic model:', best_name, flush=True)
print('TRAINING RMSE (required, fitted-data estimate):', training_metrics['RMSE'],
      flush=True)
print('OUT-OF-FOLD VALIDATION:', validation_metrics, flush=True)

## Submission contract and report

Refit the selected model on all training rows, display the mandatory training RMSE and separate validation results, and generate one output for every current test identifier. A successful CSV check does not establish the undisclosed hiring threshold.


In [ ]:
def make_submission(test_frame, prediction):
    prediction = np.asarray(prediction, dtype=float)
    if prediction.shape != (len(test_frame),) or not np.isfinite(prediction).all():
        raise ValueError('Need one finite prediction per current test identifier.')
    if test_frame.filename.duplicated().any():
        raise ValueError('Test identifiers must be unique.')
    result = pd.DataFrame({'filename': test_frame.filename.to_numpy(),
                           'label': np.clip(prediction, 0, 5)})
    assert list(result.columns) == ['filename', 'label']
    assert result.filename.tolist() == test_frame.filename.tolist()
    return result


submission = make_submission(test, test_prediction)
submission.to_csv('/kaggle/working/submission.csv', index=False)
submission.to_csv(OUT / 'submission_acoustic.csv', index=False)
joblib.dump(best_model, OUT / 'acoustic_model.joblib')
summary = {'stage': 'acoustic baseline', 'selected_model': best_name,
           'training_metrics': training_metrics, 'validation_metrics': validation_metrics,
           'validation_design': validation_type, 'seed': SEED,
           'submission_rows': len(submission), 'sample_identifiers_are_stale': True,
           'limitations': ['Acoustic cues alone do not directly measure syntax.',
                          'Model selection uses these validation folds; results '
                          'are development estimates, not an untouched final test.',
                          'Speaker/prompt groups are unavailable in the CSV; '
                          'exact-file grouping does not rule out speaker overlap.']}
(OUT / 'run_summary.json').write_text(json.dumps(summary, indent=2))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(y, bins=np.arange(-.25, 5.51, .5), color='#3b82f6')
axes[0].set(title='Training score distribution', xlabel='SHL grammar score', ylabel='Count')
axes[1].scatter(y, best_oof, alpha=.4, s=15)
axes[1].plot([0, 5], [0, 5], '--', color='black')
axes[1].set(title='Predictions on held-out folds', xlabel='True score', ylabel='Prediction')
axes[2].barh(scores.model, scores.RMSE, color='#10b981')
axes[2].invert_yaxis()
axes[2].set(title='Model comparison', xlabel='Out-of-fold RMSE (lower is better)')
fig.tight_layout()
fig.savefig(OUT / 'validation_plots.png', dpi=160)
plt.show()
print('Saved submission.csv:', len(submission), 'rows; columns filename,label.', flush=True)
print('This is a measured baseline. Add and validate transcript features next.', flush=True)

## Transcript experiment prerequisites

The remaining cells require internet access and pretrained model downloads. A GPU is recommended. If unavailable, stop after the acoustic report: its CSV remains usable, but the transcript experiment is not evaluated.


## Transcribe complete recordings

Enable internet and a GPU in Kaggle. Download the fixed pretrained Whisper model. Inference runs here, without uploading recordings to a transcription API. Cache transcripts privately and resume from checkpoints. ASR can omit grammatical errors or hallucinate speech; examine representative outputs privately. Never publish these caches.


In [ ]:
import gc
import re
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

ASR_MODEL = 'openai/whisper-small.en'
TEXT_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
gpu = torch.cuda.is_available()
torch.manual_seed(SEED)
torch.set_num_threads(2)
transcript_file = OUT / 'transcripts_private.json'
records = json.loads(transcript_file.read_text()) if transcript_file.exists() else {}
asr = None
started = time.time()
for i, ((split, name), path) in enumerate(zip(manifest, paths), 1):
    key = split + '/' + name
    if key in records and records[key].get('model') == ASR_MODEL:
        continue
    if asr is None:
        asr = pipeline('automatic-speech-recognition', model=ASR_MODEL,
                       device=0 if gpu else -1,
                       torch_dtype=torch.float16 if gpu else torch.float32,
                       chunk_length_s=30, stride_length_s=(5, 5))
    # Preserve what was spoken. Do not grammar-correct the transcript.
    result = asr({'array': read_audio(path), 'sampling_rate': SR},
                 batch_size=8 if gpu else 1,
                 generate_kwargs={'do_sample': False, 'num_beams': 1})
    records[key] = {'model': ASR_MODEL, 'text': result['text'].strip()}
    # Atomic checkpoints allow interruption and continuation.
    temporary = transcript_file.with_suffix('.tmp')
    temporary.write_text(json.dumps(records, ensure_ascii=False))
    temporary.replace(transcript_file)
    if i % 10 == 0 or i == 1:
        print(f'Transcribed {i}/{len(paths)}; {time.time()-started:.0f}s', flush=True)
del asr
gc.collect()
if gpu:
    torch.cuda.empty_cache()
texts = [records[split+'/'+name]['text'] for split, name in manifest]
print('Transcript count:', len(texts), '; empty:', sum(not t for t in texts), flush=True)

## Fixed pretrained text embeddings and speech descriptors

MiniLM encodes text semantics, not grammatical correctness. Split long transcripts into token windows and average by window length to cover the entire recording. Add word count, speech rate, lexical diversity and repetition descriptors. No pretrained encoder is fine-tuned on SHL labels.


In [ ]:
encoder = SentenceTransformer(TEXT_MODEL, device='cuda' if gpu else 'cpu')
chunks, owner, weights = [], [], []
for i, text in enumerate(texts):
    tokens = encoder.tokenizer.encode(text, add_special_tokens=False)
    # MiniLM truncates long inputs by default. Cover the entire transcript.
    for start in range(0, max(1, len(tokens)), 220):
        part = tokens[start:start+220]
        chunks.append(encoder.tokenizer.decode(part) if part else '')
        owner.append(i)
        weights.append(max(1, len(part)))
encoded = encoder.encode(chunks, batch_size=32 if gpu else 8,
                         normalize_embeddings=True, show_progress_bar=True)
embeddings = np.zeros((len(texts), encoded.shape[1]), dtype=np.float32)
totals = np.zeros(len(texts), dtype=np.float32)
for vector, i, weight in zip(encoded, owner, weights):
    embeddings[i] += vector * weight
    totals[i] += weight
embeddings /= totals[:, None]
del encoder, encoded
gc.collect()
if gpu:
    torch.cuda.empty_cache()

def text_descriptors(text, duration):
    words = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text.lower())
    n = len(words)
    filler = sum(w in {'um', 'uh', 'erm', 'hmm'} for w in words)
    repeats = sum(a == b for a, b in zip(words, words[1:]))
    sentences = max(1, len(re.findall(r'[.!?]+', text)))
    return [n, n/max(duration, 1e-6)*60, len(set(words))/max(n, 1),
            np.mean([len(w) for w in words]) if words else 0,
            n/sentences, filler/max(n, 1), repeats/max(n, 1), float(n == 0)]

descriptors = np.array([text_descriptors(t, float(f[-5]))
                        for t, f in zip(texts, all_features)], dtype=np.float32)
fixed_text = np.hstack([embeddings, descriptors])
fixed_fusion = np.hstack([all_features, fixed_text])
assert np.isfinite(fixed_fusion).all()
np.savez_compressed(OUT/'text_features_private.npz',
                    embeddings=embeddings, descriptors=descriptors,
                    manifest_id=manifest_id)

## Compare acoustic, text and combined models on the same held-out folds

Compare linguistic and acoustic features using the same five folds. Fit all scaling and TF-IDF transformations on each training fold. Select by development RMSE, refit on all supplied training labels, and write a validated CSV. Keep the acoustic model if additions do not improve validation. These development results do not establish the private hiring threshold.


In [ ]:
# The encoder is fixed, so extracting embeddings before CV does not learn labels.
# Scaling, TF-IDF vocabulary/IDF and supervised regression are learned in-fold.
candidates = {
    'Text Ridge 10': (fixed_text, make_pipeline(StandardScaler(), Ridge(alpha=10))),
    'Text Ridge 100': (fixed_text, make_pipeline(StandardScaler(), Ridge(alpha=100))),
    'Combined Ridge 100': (fixed_fusion, make_pipeline(StandardScaler(), Ridge(alpha=100))),
    'Combined SVR': (fixed_fusion, make_pipeline(StandardScaler(), SVR(C=10, epsilon=.15))),
}
extended = comparison.copy()
full_models = {}
all_oof = dict(oof_predictions)
with threadpool_limits(limits=1):
    for name, (features, estimator) in candidates.items():
        oof = np.full(len(y), np.nan)
        fold_errors = []
        for fit_ids, valid_ids in folds:
            fitted = clone(estimator).fit(features[fit_ids], y[fit_ids])
            p = np.clip(fitted.predict(features[valid_ids]), 0, 5)
            oof[valid_ids] = p
            fold_errors.append(metrics(y[valid_ids], p)['RMSE'])
        row = {'model': name, **metrics(y, oof),
               'fold_RMSE_sd': float(np.std(fold_errors))}
        print(row, flush=True)
        extended.append(row)
        all_oof[name] = oof
        full_models[name] = (features, clone(estimator).fit(features[:len(y)], y))

def fit_lexical(fit_ids, predict_ids):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=2500,
                           sublinear_tf=True)
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                           max_features=5000, sublinear_tf=True)
    scale = StandardScaler()
    fit_texts = [texts[i] for i in fit_ids]
    pred_texts = [texts[i] for i in predict_ids]
    a = hstack([word.fit_transform(fit_texts), char.fit_transform(fit_texts),
                csr_matrix(scale.fit_transform(fixed_fusion[fit_ids]))], format='csr')
    b = hstack([word.transform(pred_texts), char.transform(pred_texts),
                csr_matrix(scale.transform(fixed_fusion[predict_ids]))], format='csr')
    fitted = Ridge(alpha=100, solver='lsqr').fit(a, y[fit_ids])
    return fitted.predict(b), (word, char, scale, fitted), fitted.predict(a)

with threadpool_limits(limits=1):
    lexical_oof = np.full(len(y), np.nan)
    fold_errors = []
    for fit_ids, valid_ids in folds:
        p, _, _ = fit_lexical(fit_ids, valid_ids)
        p = np.clip(p, 0, 5)
        lexical_oof[valid_ids] = p
        fold_errors.append(metrics(y[valid_ids], p)['RMSE'])
    row = {'model': 'Combined lexical Ridge', **metrics(y, lexical_oof),
           'fold_RMSE_sd': float(np.std(fold_errors))}
    extended.append(row)
    all_oof[row['model']] = lexical_oof
    print(row, flush=True)
all_scores = pd.DataFrame(extended).sort_values('RMSE').reset_index(drop=True)
all_scores.to_csv(OUT/'all_validation_metrics.csv', index=False)
selected_name = all_scores.iloc[0]['model']
with threadpool_limits(limits=1):
    if selected_name == 'Combined lexical Ridge':
        final_test, selected_model, final_train = fit_lexical(
            np.arange(len(y)), np.arange(len(y), len(texts)))
    elif selected_name in full_models:
        features, selected_model = full_models[selected_name]
        final_train = selected_model.predict(features[:len(y)])
        final_test = selected_model.predict(features[len(y):])
    else:
        selected_model = clone(models[selected_name]).fit(X, y)
        final_train = selected_model.predict(X)
        final_test = selected_model.predict(X_test)
final_train, final_test = np.clip(final_train, 0, 5), np.clip(final_test, 0, 5)
final_submission = make_submission(test, final_test)
final_submission.to_csv('/kaggle/working/submission.csv', index=False)
joblib.dump(selected_model, OUT/'selected_model.joblib')
pd.DataFrame({'target': y, 'prediction': all_oof[selected_name]}).to_csv(
    OUT/'oof_predictions_private.csv', index=False)
final_summary = {
    'selected_model': selected_name, 'asr_model': ASR_MODEL, 'text_model': TEXT_MODEL,
    'training_metrics': metrics(y, final_train),
    'validation_metrics': metrics(y, all_oof[selected_name]),
    'validation_design': validation_type, 'test_rows': len(test),
    'limitations': ['ASR may omit errors or hallucinate words.',
                   'Semantic features may encode topic instead of grammar.',
                   'Model selection uses development CV; it is not an untouched test.',
                   'Unavailable speaker/prompt identifiers limit leakage checks.']}
(OUT/'final_summary.json').write_text(json.dumps(final_summary, indent=2))
print('FINAL MEASURED REPORT:', json.dumps(final_summary, indent=2), flush=True)
display(all_scores)
plt.figure(figsize=(5, 4))
plt.scatter(y, all_oof[selected_name], alpha=.4, s=15)
plt.plot([0, 5], [0, 5], '--', color='black')
plt.xlabel('True score'); plt.ylabel('Out-of-fold prediction')
plt.title(selected_name); plt.tight_layout()
plt.savefig(OUT/'final_validation_plot.png', dpi=160); plt.show()
print('Validated submission.csv:', len(final_submission), 'rows.', flush=True)

## Pretrained speech representations

Reuse the fixed Whisper encoder to summarize learned speech representations over complete recordings, covering each 30-second window. Exclude padded time positions from summary statistics. Keep feature caches private. This is an additional measured experiment, not evidence of a better score until evaluated.


In [ ]:
from transformers import AutoFeatureExtractor, WhisperForConditionalGeneration

speech_cache = OUT/'whisper_features_private.npz'
speech_cache_id = hashlib.sha256((manifest_id + ASR_MODEL + ':masked-moments-v1').encode()).hexdigest()
speech_rows = []
if speech_cache.exists():
    saved = np.load(speech_cache, allow_pickle=False)
    if str(saved['cache_id']) != speech_cache_id:
        raise ValueError('Speech cache configuration does not match.')
    speech_rows = list(saved['features'])
assert len(speech_rows) <= len(paths)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
if len(speech_rows) < len(paths):
    extractor = AutoFeatureExtractor.from_pretrained(ASR_MODEL)
    speech_encoder = WhisperForConditionalGeneration.from_pretrained(
        ASR_MODEL, torch_dtype=dtype).get_encoder().to(device).eval()
    started = time.time()
    for i in range(len(speech_rows), len(paths)):
        waveform = read_audio(paths[i])
        segments = [waveform[start:start+30*SR]
                    for start in range(0, len(waveform), 30*SR)]
        total, sum_vector, sum_squares = 0, None, None
        for start in range(0, len(segments), 4):
            batch = segments[start:start+4]
            inputs = extractor(batch, sampling_rate=SR, return_tensors='pt').input_features
            with torch.inference_mode():
                hidden = speech_encoder(inputs.to(device=device, dtype=dtype)).last_hidden_state
            # Whisper outputs one frame per 20 ms. Exclude padded time positions.
            hidden = hidden.float().cpu().numpy()
            for segment, frames in zip(batch, hidden):
                n_valid = min(len(frames), max(1, math.ceil(len(segment)/(2*extractor.hop_length))))
                frames = frames[:n_valid].astype(np.float64)
                if sum_vector is None:
                    sum_vector = np.zeros(frames.shape[1], dtype=np.float64)
                    sum_squares = np.zeros_like(sum_vector)
                total += len(frames)
                sum_vector += frames.sum(axis=0)
                sum_squares += (frames**2).sum(axis=0)
        mean = sum_vector/total
        standard_deviation = np.sqrt(np.maximum(0, sum_squares/total-mean**2))
        speech_rows.append(np.concatenate([mean, standard_deviation]).astype(np.float32))
        if (i+1) % 50 == 0 or i+1 == len(paths):
            np.savez_compressed(speech_cache, features=np.vstack(speech_rows),
                                cache_id=speech_cache_id)
            print(f'Speech representations {i+1}/{len(paths)}; '
                  f'{time.time()-started:.0f}s', flush=True)
    del speech_encoder, extractor
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
speech_features = np.vstack(speech_rows)
assert np.isfinite(speech_features).all()
speech_fusion = np.hstack([speech_features, fixed_fusion])

## Evaluate speech representations and retain the best measured model

Compare speech-only Ridge and SVR, and combined speech/text/acoustic SVR, on the same development folds. Keep an earlier model if it performs better. The final printed report and submission file reflect the lowest development RMSE across the measured candidates.


In [ ]:
speech_candidates = {
    'Speech Ridge': (speech_features, make_pipeline(StandardScaler(), Ridge(alpha=100))),
    'Speech SVR': (speech_features, make_pipeline(StandardScaler(), SVR(C=10, epsilon=.15))),
    'All features SVR': (speech_fusion, make_pipeline(StandardScaler(), SVR(C=10, epsilon=.15))),
}
speech_comparison = all_scores.to_dict('records')
speech_models = {}
with threadpool_limits(limits=1):
    for name, (features, estimator) in speech_candidates.items():
        oof = np.full(len(y), np.nan)
        fold_errors = []
        for fit_ids, valid_ids in folds:
            fitted = clone(estimator).fit(features[fit_ids], y[fit_ids])
            p = np.clip(fitted.predict(features[valid_ids]), 0, 5)
            oof[valid_ids] = p
            fold_errors.append(metrics(y[valid_ids], p)['RMSE'])
        assert np.isfinite(oof).all()
        row = {'model': name, **metrics(y, oof),
               'fold_RMSE_sd': float(np.std(fold_errors))}
        speech_comparison.append(row)
        all_oof[name] = oof
        speech_models[name] = (features, clone(estimator).fit(features[:len(y)], y))
        print(row, flush=True)
all_scores = pd.DataFrame(speech_comparison).sort_values('RMSE').reset_index(drop=True)
all_scores.to_csv(OUT/'all_validation_metrics.csv', index=False)
selected_name = all_scores.iloc[0]['model']
if selected_name in speech_models:
    features, selected_model = speech_models[selected_name]
    final_train = np.clip(selected_model.predict(features[:len(y)]), 0, 5)
    final_test = np.clip(selected_model.predict(features[len(y):]), 0, 5)
    final_submission = make_submission(test, final_test)
    final_submission.to_csv('/kaggle/working/submission.csv', index=False)
    joblib.dump(selected_model, OUT/'selected_model.joblib')
    final_summary.update(selected_model=selected_name,
                         training_metrics=metrics(y, final_train),
                         validation_metrics=metrics(y, all_oof[selected_name]),
                         speech_feature_model=ASR_MODEL)
    (OUT/'final_summary.json').write_text(json.dumps(final_summary, indent=2))
pd.DataFrame({'target': y, 'prediction': all_oof[selected_name]}).to_csv(
    OUT/'oof_predictions_private.csv', index=False)
print('FINAL MEASURED REPORT:', json.dumps(final_summary, indent=2), flush=True)
display(all_scores)
plt.figure(figsize=(5, 4))
plt.scatter(y, all_oof[selected_name], alpha=.4, s=15)
plt.plot([0, 5], [0, 5], '--', color='black')
plt.xlabel('True score'); plt.ylabel('Out-of-fold prediction')
plt.title(selected_name); plt.tight_layout()
plt.savefig(OUT/'final_validation_plot.png', dpi=160); plt.show()
print('Validated submission.csv:', len(final_submission), 'rows.', flush=True)

## Final measured report and private export

Verify every output row against current test.csv. Record package versions and print the actual selected model's training and validation metrics. The following output contains competition predictions and must remain private. Clear executed outputs before publishing code.


In [ ]:
"""Private final report and CSV export, after all model experiments."""
import importlib.metadata
import zipfile

checked = pd.read_csv('/kaggle/working/submission.csv')
assert list(checked.columns) == ['filename', 'label']
assert checked.filename.tolist() == test.filename.tolist()
assert checked.filename.is_unique
assert np.isfinite(checked.label).all() and checked.label.between(0, 5).all()
versions = {name: importlib.metadata.version(name) for name in [
    'numpy', 'pandas', 'scipy', 'scikit-learn', 'librosa', 'soundfile',
    'torch', 'transformers', 'sentence-transformers']}
(OUT/'environment_versions.json').write_text(json.dumps(versions, indent=2))
report = (
    'SHL Grammar Scoring: measured development report\n\n'
    f"Selected model: {final_summary['selected_model']}\n"
    f"Training RMSE (fitted training records): {final_summary['training_metrics']['RMSE']:.6f}\n"
    f"Out-of-fold RMSE: {final_summary['validation_metrics']['RMSE']:.6f}\n"
    f"Out-of-fold Pearson: {final_summary['validation_metrics']['Pearson']:.6f}\n"
    f"Validation: {validation_type}.\n"
    f"Training records: {len(train)}. Test predictions: {len(checked)}.\n\n"
    'Method: complete-recording acoustic statistics; fixed pretrained Whisper\n'
    'transcription and speech representations; MiniLM text representations;\n'
    'linguistic descriptors and in-fold TF-IDF. Supervised regressors use only\n'
    'the supplied training labels. Scaling and learned text vocabulary are\n'
    'fitted inside folds. The selected model is refitted on all training labels.\n\n'
    'The supplied sample contains stale identifiers. The output instead\n'
    'matches all current test.csv rows and retains filename,label columns.\n\n'
    'Limitations: model selection uses development folds, not an independent\n'
    'test. Speaker/prompt identifiers are unavailable. ASR can omit errors or\n'
    'hallucinate words, semantics can encode topic, and acoustic cues may\n'
    'capture recording conditions. No private hiring threshold is assumed.\n'
    'Private recordings, transcripts, labels and caches are not public code.\n')
(OUT/'final_report.txt').write_text(report)
with zipfile.ZipFile('/kaggle/working/shl_final_results_private.zip', 'w',
                     compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write('/kaggle/working/submission.csv', 'submission.csv')
    for name in ['final_summary.json', 'all_validation_metrics.csv',
                 'final_report.txt', 'environment_versions.json',
                 'final_validation_plot.png']:
        bundle.write(OUT/name, name)
print(report)
print('SHL_EXPORT_SUMMARY_BEGIN')
print(json.dumps(final_summary))
print('SHL_EXPORT_SUMMARY_END')
print('SHL_EXPORT_CSV_BEGIN')
print(checked.to_csv(index=False), end='')
print('SHL_EXPORT_CSV_END')

## Brief report and interview discussion

The completed run on 7 October 2026 selected **All features SVR** from 13 measured candidates. On 769 training recordings, fitted-training RMSE was **0.1365453254**. Five-fold out-of-fold RMSE was **0.5617009970**, with Pearson correlation **0.8994185092**. These development estimates are separate from a Kaggle leaderboard score. All 216 test predictions passed identifier, ordering, finiteness and 0-5 range checks.

The acoustic SVR reference achieved OOF RMSE 0.69642. Text-only Ridge achieved 1.03928; adding text alone did not improve the acoustic reference. Fixed Whisper speech representations gave speech-only SVR RMSE 0.56537, providing the main improvement. Combining speech, acoustic and text features improved it slightly to 0.56170. No pretrained encoder was fine-tuned on SHL labels, and no external labeled grammar dataset was added.

Acoustic features may capture pronunciation, fluency or recording conditions. ASR may omit or normalize grammatical errors, and semantic embeddings may encode topic. Model selection used these same development folds, so results are not an untouched test estimate. Speaker and prompt identifiers were unavailable, limiting leakage checks. We preserve ASR transcripts rather than rewriting them into correct English.

The public repository contains source code and instructions only. Competition audio, transcripts, labeled metadata, predictions and executed outputs remain private.
